# Solution: Level 2 — Intermediate: Cleaning, Selecting, Grouping and Merging

**Course:** Python & Data Analytics &nbsp;|&nbsp; **Program:** B.Tech Technical Training – CSE A & B

**What this set covers:** finding and fixing data-quality problems (missing values, duplicates, inconsistent
text, wrong data types, impossible values), selecting data with `.loc` / `.iloc`, feature creation,
`groupby` aggregation, pivot tables and merging DataFrames.

Most of this set uses `sales_data.csv`. The last part uses `student_performance.csv`.
The raw files contain **deliberate data-quality problems**. Finding them is part of the exercise.

> **Rule for this level:** never edit the CSV file by hand. Every fix must be done in code so it can be repeated.

> **Instructor solution.** Attempt the exercises yourself before reading this. Your answers may differ in method and still be correct.

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "../../datasets/"
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

## Part A — Inspecting raw data

### Exercise 2.1 — First inspection

Load `sales_data.csv` into `sales`. Use `head()`, `info()` and `describe(include='all')` to look at it. In a markdown cell, list at least **three** things that look wrong or suspicious.

In [2]:
sales = pd.read_csv(DATA_DIR + "sales_data.csv")
display(sales.head())
sales.info()
display(sales.describe(include="all").T)
print("Distinct Region values:", sales["Region"].unique())

,Order_ID,Order_Date,Region,City,Customer_Segment,Category,Product,Quantity,Unit_Price,Discount,Payment_Mode
0,ORD10069,2024-01-01,North,Delhi,Consumer,Clothing,Jeans,2.00,1412.69,0.05,Credit Card
1,ORD10125,2024-01-02,North,Chandigarh,Consumer,Electronics,Headphones,1.00,2362.65,0.15,Cash
2,ORD11101,2024-01-02,South,Hyderabad,Corporate,Electronics,Smartphone,3.00,17702.51,0.05,Credit Card
3,ORD10395,2024-01-03,Central,Nagpur,Corporate,Electronics,Headphones,8.00,2368.19,0.05,Debit Card
4,ORD11161,2024-01-03,East,Bhubaneswar,Corporate,Electronics,Smartphone,2.00,16847.36,0.00,UPI


<class 'pandas.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          1215 non-null   str    
 1   Order_Date        1215 non-null   str    
 2   Region            1215 non-null   str    
 3   City              1215 non-null   str    
 4   Customer_Segment  1205 non-null   str    
 5   Category          1215 non-null   str    
 6   Product           1215 non-null   str    
 7   Quantity          1195 non-null   float64
 8   Unit_Price        1203 non-null   str    
 9   Discount          1215 non-null   float64
 10  Payment_Mode      1215 non-null   str    
dtypes: float64(2), str(9)
memory usage: 104.5 KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,1215,1200,ORD11069,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,1215,348,2024-08-07,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,1215,19,South,343,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,1215,15,Hyderabad,129,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Segment,1205,3,Consumer,670,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,1215,4,Electronics,389,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,1215,18,Headphones,95,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,"1,195.00",NaN,NaN,NaN,3.93,2.83,1.00,2.00,3.00,6.00,10.00
Unit_Price,1203,1183,Rs.6543.15,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Discount,"1,215.00",NaN,NaN,NaN,0.07,0.06,0.00,0.00,0.05,0.10,0.20


Distinct Region values: <StringArray>
[   'North',    'South',  'Central',     'East',     'West',   'South ',
 'Central ',    'SOUTH',    'south',     'west',    ' West',   ' South',
    'north',     'WEST',   ' North',  'CENTRAL',    'West ',    'NORTH',
    'East ']
Length: 19, dtype: str


> **Note:** Problems found: (1) `Unit_Price` is text rather than a number because some prices are stored as `'Rs.1234.5'`; (2) `Region` has inconsistent spelling, case and spacing (`'south'`, `'NORTH'`, `' West'`); (3) `Quantity`, `Unit_Price` and `Customer_Segment` have missing values; (4) there are duplicate rows; (5) `Order_Date` is stored as text, not as a date.

### Exercise 2.2 — Measure the missing values

Build a small table showing, for each column, the **count** and **percentage** of missing values. Show only the columns that have at least one missing value.

In [3]:
missing = pd.DataFrame({
    "missing_count": sales.isna().sum(),
    "missing_pct": (sales.isna().mean() * 100).round(2),
})
missing[missing["missing_count"] > 0]

,missing_count,missing_pct
Customer_Segment,10,0.82
Quantity,20,1.65
Unit_Price,12,0.99


### Exercise 2.3 — Duplicates

How many rows are exact duplicates? Display a few of them (show every copy, not just the repeats), then remove them and confirm the new number of rows. Also check: after removing exact duplicates, is `Order_ID` unique?

In [4]:
print("Exact duplicate rows:", sales.duplicated().sum())
display(sales[sales.duplicated(keep=False)].sort_values("Order_ID").head(6))

before = len(sales)
sales = sales.drop_duplicates().reset_index(drop=True)
print(f"Rows: {before} -> {len(sales)}")
print("Order_ID unique now?", sales["Order_ID"].is_unique)

Exact duplicate rows: 15


,Order_ID,Order_Date,Region,City,Customer_Segment,Category,Product,Quantity,Unit_Price,Discount,Payment_Mode
863,ORD10025,2024-10-11,South,Chennai,Consumer,Clothing,Jacket,1.00,2664.42,0.20,UPI
858,ORD10025,2024-10-11,South,Chennai,Consumer,Clothing,Jacket,1.00,2664.42,0.20,UPI
104,ORD10050,2024-02-05,West,Pune,Corporate,Electronics,Smartwatch,3.00,6870.14,0.20,UPI
108,ORD10050,2024-02-05,West,Pune,Corporate,Electronics,Smartwatch,3.00,6870.14,0.20,UPI
867,ORD10367,2024-10-12,North,Chandigarh,Corporate,Furniture,Study Table,2.00,9221.55,0.20,Debit Card
865,ORD10367,2024-10-12,North,Chandigarh,Corporate,Furniture,Study Table,2.00,9221.55,0.20,Debit Card


Rows: 1215 -> 1200
Order_ID unique now? True


## Part B — Cleaning

### Exercise 2.4 — Fix inconsistent text

Standardise `Region` so that there are exactly five values: North, South, East, West and Central. Show `value_counts()` before and after.

In [5]:
print(sales["Region"].value_counts(), end="\n\n")
sales["Region"] = sales["Region"].str.strip().str.title()
print(sales["Region"].value_counts())

Region
South       336
West        259
North       247
East        195
Central     123
SOUTH         6
 West         5
Central       4
 South        4
north         4
South         3
west          3
WEST          3
south         2
CENTRAL       2
 North        1
West          1
NORTH         1
East          1
Name: count, dtype: int64

Region
South      351
West       271
North      253
East       196
Central    129
Name: count, dtype: int64


### Exercise 2.5 — Fix a numeric column stored as text

Convert `Unit_Price` to a numeric (float) column. First remove the `'Rs.'` prefix. Then use `pd.to_numeric` with `errors='coerce'`. Afterwards, confirm the data type and that no extra missing values were created by the conversion.

In [6]:
missing_before = sales["Unit_Price"].isna().sum()
sales["Unit_Price"] = pd.to_numeric(
    sales["Unit_Price"].astype(str).str.replace("Rs.", "", regex=False).str.strip(),
    errors="coerce",
)
print("dtype:", sales["Unit_Price"].dtype)
print("Missing before / after:", missing_before, "/", sales["Unit_Price"].isna().sum())

dtype: float64
Missing before / after: 12 / 12


> **Note:** `errors='coerce'` turns anything that cannot be parsed into `NaN` and does not raise an error. That is convenient, but it can hide problems. Always compare the missing count before and after.

### Exercise 2.6 — Handle missing values sensibly

Decide how to fill each kind of missing value, and explain why in a markdown cell:
- `Unit_Price`: fill with the **median price of the same product** (prices differ a lot between products)
- `Quantity`: fill with the **median quantity of the same product**
- `Customer_Segment`: fill with `'Unknown'` (we should not guess a customer's type)

Confirm that no missing values remain.

In [7]:
for col in ["Unit_Price", "Quantity"]:
    sales[col] = sales[col].fillna(sales.groupby("Product")[col].transform("median"))

sales["Customer_Segment"] = sales["Customer_Segment"].fillna("Unknown")
sales["Quantity"] = sales["Quantity"].round().astype(int)   # quantities are whole numbers

print(sales.isna().sum().sum(), "missing values remain")

0 missing values remain


> **Note:** Filling with the overall mean price would be wrong here. A missing laptop price would get the average of laptops and pens together. A group-wise median is robust to outliers and respects the structure of the data.

### Exercise 2.7 — Work with dates

Convert `Order_Date` to a datetime type. Add the columns `Month` (1–12), `Month_Name` (Jan, Feb, …), `Quarter` and `Day_of_Week`. What is the first and last order date in the data?

In [8]:
sales["Order_Date"] = pd.to_datetime(sales["Order_Date"])
sales["Month"] = sales["Order_Date"].dt.month
sales["Month_Name"] = sales["Order_Date"].dt.strftime("%b")
sales["Quarter"] = sales["Order_Date"].dt.quarter
sales["Day_of_Week"] = sales["Order_Date"].dt.day_name()

print("From", sales["Order_Date"].min().date(), "to", sales["Order_Date"].max().date())
sales[["Order_Date", "Month", "Month_Name", "Quarter", "Day_of_Week"]].head()

From 2024-01-01 to 2024-12-31


,Order_Date,Month,Month_Name,Quarter,Day_of_Week
0,2024-01-01,1,Jan,1,Monday
1,2024-01-02,1,Jan,1,Tuesday
2,2024-01-02,1,Jan,1,Tuesday
3,2024-01-03,1,Jan,1,Wednesday
4,2024-01-03,1,Jan,1,Wednesday


### Exercise 2.8 — Create a calculated column

Create `Revenue = Quantity × Unit_Price × (1 − Discount)`, rounded to 2 decimals. What is the total revenue for the year, in lakhs of rupees? (1 lakh = 1,00,000)

In [9]:
sales["Revenue"] = (sales["Quantity"] * sales["Unit_Price"] * (1 - sales["Discount"])).round(2)
print(f"Total revenue: Rs. {sales['Revenue'].sum() / 1e5:,.2f} lakh")
sales[["Product", "Quantity", "Unit_Price", "Discount", "Revenue"]].head()

Total revenue: Rs. 219.01 lakh


,Product,Quantity,Unit_Price,Discount,Revenue
0,Jeans,2,"1,412.69",0.05,"2,684.11"
1,Headphones,1,"2,362.65",0.15,"2,008.25"
2,Smartphone,3,"17,702.51",0.05,"50,452.15"
3,Headphones,8,"2,368.19",0.05,"17,998.24"
4,Smartphone,2,"16,847.36",0.00,"33,694.72"


## Part C — Selecting with `.loc` and `.iloc`

### Exercise 2.9 — Label-based selection with .loc

Using `.loc`, show the `Order_ID`, `City`, `Product` and `Revenue` of orders that are:
1. from the **South** region with revenue above Rs. 50,000
2. in the **Electronics** category, paid by **UPI**, in **October or November**

How many orders match each condition?

In [10]:
cols = ["Order_ID", "City", "Product", "Revenue"]

south_big = sales.loc[(sales["Region"] == "South") & (sales["Revenue"] > 50_000), cols]
print(len(south_big), "South orders above Rs. 50,000")
display(south_big.head())

festive_upi = sales.loc[
    (sales["Category"] == "Electronics")
    & (sales["Payment_Mode"] == "UPI")
    & (sales["Month"].isin([10, 11])),
    cols,
]
print(len(festive_upi), "festive-season Electronics orders paid by UPI")

26 South orders above Rs. 50,000


,Order_ID,City,Product,Revenue
2,ORD11101,Hyderabad,Smartphone,"50,452.15"
85,ORD11105,Bengaluru,Smartphone,"51,143.49"
91,ORD10938,Bengaluru,Laptop,"50,720.55"
115,ORD11170,Hyderabad,Sofa,"66,833.92"
161,ORD10409,Chennai,Laptop,"111,899.36"


35 festive-season Electronics orders paid by UPI


### Exercise 2.10 — Position-based selection with .iloc

Using `.iloc` only:
1. the first 5 rows
2. rows 10 to 14 and the first 4 columns
3. the very last row
4. every 100th row (rows 0, 100, 200, …), showing only `Order_ID` and `Revenue`

In [11]:
display(sales.iloc[:5])
display(sales.iloc[10:15, :4])
display(sales.iloc[[-1]])

col_pos = [sales.columns.get_loc("Order_ID"), sales.columns.get_loc("Revenue")]
display(sales.iloc[::100, col_pos])

,Order_ID,Order_Date,Region,City,Customer_Segment,Category,Product,Quantity,Unit_Price,Discount,Payment_Mode,Month,Month_Name,Quarter,Day_of_Week,Revenue
0,ORD10069,2024-01-01,North,Delhi,Consumer,Clothing,Jeans,2,"1,412.69",0.05,Credit Card,1,Jan,1,Monday,"2,684.11"
1,ORD10125,2024-01-02,North,Chandigarh,Consumer,Electronics,Headphones,1,"2,362.65",0.15,Cash,1,Jan,1,Tuesday,"2,008.25"
2,ORD11101,2024-01-02,South,Hyderabad,Corporate,Electronics,Smartphone,3,"17,702.51",0.05,Credit Card,1,Jan,1,Tuesday,"50,452.15"
3,ORD10395,2024-01-03,Central,Nagpur,Corporate,Electronics,Headphones,8,"2,368.19",0.05,Debit Card,1,Jan,1,Wednesday,"17,998.24"
4,ORD11161,2024-01-03,East,Bhubaneswar,Corporate,Electronics,Smartphone,2,"16,847.36",0.00,UPI,1,Jan,1,Wednesday,"33,694.72"


,Order_ID,Order_Date,Region,City
10,ORD10635,2024-01-04,West,Pune
11,ORD11019,2024-01-04,South,Hyderabad
12,ORD10239,2024-01-05,South,Hyderabad
13,ORD10763,2024-01-05,West,Ahmedabad
14,ORD10415,2024-01-07,South,Hyderabad


,Order_ID,Order_Date,Region,City,Customer_Segment,Category,Product,Quantity,Unit_Price,Discount,Payment_Mode,Month,Month_Name,Quarter,Day_of_Week,Revenue
1199,ORD10955,2024-12-31,West,Pune,Consumer,Office Supplies,Whiteboard,6,"1,504.49",0.15,Net Banking,12,Dec,4,Tuesday,"7,672.90"


,Order_ID,Revenue
0,ORD10069,"2,684.11"
100,ORD10908,"7,063.92"
200,ORD11097,"25,752.42"
300,ORD10190,"18,122.31"
400,ORD10315,"48,932.53"
500,ORD10962,"29,531.26"
600,ORD10396,"8,958.20"
700,ORD10223,"14,742.10"
800,ORD10001,"12,152.88"
900,ORD10674,"3,636.03"


> **Note:** Remember: `.loc` uses **labels** and the end of a slice is *included*. `.iloc` uses **positions** and the end is *excluded*.

## Part D — Grouping and aggregation

### Exercise 2.11 — Revenue by region

Calculate total revenue by `Region`, sorted from highest to lowest. What share (%) of total revenue does the top region contribute?

In [12]:
region_rev = sales.groupby("Region")["Revenue"].sum().sort_values(ascending=False)
display(region_rev.to_frame())
top = region_rev.index[0]
print(f"{top} contributes {region_rev.iloc[0] / region_rev.sum() * 100:.1f}% of revenue")

,Revenue
Region,
South,"6,279,563.23"
West,"5,347,080.64"
North,"4,649,076.02"
East,"3,573,708.24"
Central,"2,051,323.62"


South contributes 28.7% of revenue


### Exercise 2.12 — Several aggregations at once

For each `Category`, compute: number of orders, total units sold, total revenue, average order value and average discount. Use **named aggregation** so the result has readable column names.

In [13]:
cat_summary = (
    sales.groupby("Category")
    .agg(orders=("Order_ID", "count"),
         units=("Quantity", "sum"),
         revenue=("Revenue", "sum"),
         avg_order_value=("Revenue", "mean"),
         avg_discount=("Discount", "mean"))
    .sort_values("revenue", ascending=False)
)
cat_summary

,orders,units,revenue,avg_order_value,avg_discount
Category,,,,,
Electronics,384,1097,"13,246,417.43","34,495.88",0.07
Furniture,186,364,"4,685,475.63","25,190.73",0.07
Clothing,251,1319,"2,789,262.13","11,112.60",0.07
Office Supplies,379,1947,"1,179,596.56","3,112.39",0.07


### Exercise 2.13 — Two-way summary: pivot table

Create a table with **regions as rows** and **categories as columns**, showing total revenue in lakhs. Add row and column totals.

In [14]:
pivot = pd.pivot_table(sales, index="Region", columns="Category", values="Revenue",
                       aggfunc="sum", margins=True, margins_name="Total") / 1e5
pivot.round(2)

Category,Clothing,Electronics,Furniture,Office Supplies,Total
Region,,,,,
Central,4.32,10.24,4.46,1.49,20.51
East,3.49,23.06,7.41,1.78,35.74
North,5.81,27.93,10.00,2.75,46.49
South,6.84,42.43,10.43,3.09,62.80
West,7.43,28.80,14.56,2.69,53.47
Total,27.89,132.46,46.85,11.80,219.01


### Exercise 2.14 — Top products within each group

Find the **top 2 products by revenue in each region**.

In [15]:
prod_region = (
    sales.groupby(["Region", "Product"], as_index=False)["Revenue"].sum()
    .sort_values(["Region", "Revenue"], ascending=[True, False])
)
prod_region.groupby("Region").head(2)

,Region,Product,Revenue
4,Central,Laptop,"484,521.92"
11,Central,Smartphone,"243,268.03"
22,East,Laptop,"1,350,997.45"
31,East,Sofa,"416,549.47"
40,North,Laptop,"1,490,822.00"
49,North,Sofa,"671,626.90"
58,South,Laptop,"2,300,110.72"
65,South,Smartphone,"880,480.06"
76,West,Laptop,"1,555,054.10"
85,West,Sofa,"1,125,448.91"


## Part E — Merging DataFrames

### Exercise 2.15 — Merge with a targets table

Management has set the annual targets below. Merge them with the actual revenue by region, then calculate `Achievement_Pct = Actual / Target × 100`. Which regions missed their target?

In [16]:
targets = pd.DataFrame({
    "Region": ["North", "South", "East", "West", "Central"],
    "Regional_Manager": ["R. Mehta", "S. Reddy", "A. Das", "P. Shah", "K. Verma"],
    "Target_Revenue": [5_000_000, 6_000_000, 3_500_000, 5_500_000, 2_500_000],
})

actual = region_rev.reset_index().rename(columns={"Revenue": "Actual_Revenue"})
perf = targets.merge(actual, on="Region", how="left")
perf["Achievement_Pct"] = (perf["Actual_Revenue"] / perf["Target_Revenue"] * 100).round(1)
display(perf.sort_values("Achievement_Pct", ascending=False))
print("Missed target:", list(perf.loc[perf["Achievement_Pct"] < 100, "Region"]))

,Region,Regional_Manager,Target_Revenue,Actual_Revenue,Achievement_Pct
1,South,S. Reddy,6000000,"6,279,563.23",104.70
2,East,A. Das,3500000,"3,573,708.24",102.10
3,West,P. Shah,5500000,"5,347,080.64",97.20
0,North,R. Mehta,5000000,"4,649,076.02",93.00
4,Central,K. Verma,2500000,"2,051,323.62",82.10


Missed target: ['North', 'West', 'Central']


### Exercise 2.16 — Join types matter

The table below has city populations, but it covers only some cities and includes one city with no sales. Merge it with order counts per city using `inner`, `left` and `outer` joins. Compare the number of rows each time and explain the difference in a markdown cell.

In [17]:
city_info = pd.DataFrame({
    "City": ["Hyderabad", "Chennai", "Mumbai", "Delhi", "Kolkata", "Visakhapatnam"],
    "Population_Lakh": [105, 117, 214, 329, 150, 23],
})
city_orders = sales.groupby("City", as_index=False).agg(Orders=("Order_ID", "count"))

for how in ["inner", "left", "outer"]:
    merged = city_orders.merge(city_info, on="City", how=how)
    print(f"{how:>5} join: {len(merged)} rows")

display(city_orders.merge(city_info, on="City", how="outer", indicator=True).tail(8))

inner join: 5 rows
 left join: 15 rows
outer join: 16 rows


,City,Orders,Population_Lakh,_merge
8,Indore,32.00,NaN,left_only
9,Kolkata,66.00,150.00,both
10,Lucknow,85.00,NaN,left_only
11,Mumbai,92.00,214.00,both
12,Nagpur,44.00,NaN,left_only
13,Patna,62.00,NaN,left_only
14,Pune,94.00,NaN,left_only
15,Visakhapatnam,NaN,23.00,right_only


> **Note:** **inner** keeps only cities in both tables (5). **left** keeps every city with orders (15), and cities without population data get `NaN`. **outer** keeps everything (16), including Visakhapatnam, which has no orders. `indicator=True` adds a `_merge` column showing where each row came from.

## Part F — Apply it to the student dataset

### Exercise 2.17 — Clean the student data

Load `student_performance.csv` and:
1. remove duplicate rows
2. standardise `Gender` to exactly `Male` / `Female`
3. set impossible values to `NaN`: attendance outside 0–100 and subject marks outside 0–100
4. report how many values you changed in step 3

In [18]:
students = pd.read_csv(DATA_DIR + "student_performance.csv").drop_duplicates().reset_index(drop=True)

gender_map = {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}
students["Gender"] = students["Gender"].str.strip().str.lower().map(gender_map)
print(students["Gender"].value_counts(dropna=False), end="\n\n")

subjects = ["Mathematics", "Physics", "Programming", "English", "Electronics"]
changed = 0
for col in ["Attendance_Pct"] + subjects:
    valid = students[col].between(0, 100) | students[col].isna()
    changed += (~valid).sum()
    students[col] = students[col].where(valid)
print("Impossible values set to NaN:", changed)
print("Rows:", len(students))

Gender
Male      147
Female     93
Name: count, dtype: int64

Impossible values set to NaN: 5
Rows: 240


### Exercise 2.18 — Merge mentors and summarise by section

Each section has a faculty mentor (table below). Merge it into `students`, then produce a summary per section and mentor: number of students, average attendance and average mark in each subject.

In [19]:
mentors = pd.DataFrame({"Section": ["A", "B"],
                        "Mentor": ["Dr. Lakshmi Prasad", "Dr. Ravi Teja"]})

students = students.merge(mentors, on="Section", how="left")
summary = students.groupby(["Section", "Mentor"]).agg(
    Students=("Student_ID", "count"),
    Avg_Attendance=("Attendance_Pct", "mean"),
    **{f"Avg_{s}": (s, "mean") for s in subjects},
)
summary.round(1)

,,Students,Avg_Attendance,Avg_Mathematics,Avg_Physics,Avg_Programming,Avg_English,Avg_Electronics
Section,Mentor,,,,,,,
A,Dr. Lakshmi Prasad,120,77.30,60.50,63.90,66.10,71.30,57.70
B,Dr. Ravi Teja,120,76.70,60.80,63.80,66.10,72.00,59.50


### Exercise 2.19 — Save the cleaned data

Save the cleaned sales and student DataFrames as `sales_clean.csv` and `students_clean.csv` in this folder. Reload one of them and check that the shape matches.

In [20]:
sales.to_csv("sales_clean.csv", index=False)
students.to_csv("students_clean.csv", index=False)

check = pd.read_csv("sales_clean.csv")
print("Saved and reloaded:", check.shape, "| original:", sales.shape)

Saved and reloaded: (1200, 16) | original: (1200, 16)


---
## Key takeaways
- A cleaning step is only finished when you have **checked** it: counts before and after, `value_counts()`, `dtype`.
- Fill missing values in a way that respects the data's structure (group-wise medians), and document the decision.
- `groupby` + named `agg` covers most business summaries. `pivot_table` adds a second dimension.
- Choose the join type on purpose. Always compare row counts before and after a merge.